- Nombre: Santiago Silveira

---
## **Implementación de un Asistente Inteligente para responder preguntas sobre documentos**

### Análisis del problema

Se requiere construir un asistente que responda preguntas basadas en un conjunto de documentos, específicamente el dataset *HotpotQA*. La característica distintiva de este dataset es que requiere razonamiento **multi-hop**, lo que implica que para responder correctamente, el sistema debe conectar información de múltiples documentos.

- **hablar de los json**

![Diagrama Pipeline RAG](.\imagenes\rag-pipeline.png "Pipeline RAG")

---
### Importando las Librerías

In [21]:
# De uso general
import json
import os
from IPython.display import clear_output
from typing import List
from dotenv import load_dotenv

# Cargar variables de entorno
load_dotenv()

# Componentes de LlamaIndex
from llama_index.core import (
    Document,
    GPTVectorStoreIndex,
    Settings,
    StorageContext)
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.llms.openai import OpenAI

import chromadb

clear_output()

---
### Carga e ingestión de los datos

... para enriquecer el contexto del documento, no solo voy a incluir el campo `text` en cada documento, sino también `question`, `answer` y `title`. Incluyendo todos los campos pretendo proporcionarle al sistema más información y metadatos que pueden ser útiles para:
1. **Mejorar la relevancia en la búsqueda**: el motor de recuperación tiene más datos para comparar y relacionar con la consulta del usuario. Por ejemplo, el título o incluso la pregunta orignial del dataset pueden contener pistas importantes sobre el contenido del documento.
2. **Contextualizar la información**: Entiendo que en escenarios de *multi-hop reasoning*, la relación entre la pregunta original, la respuesta y el contenido textual clave. La combianción de estos campos ayuda a que el modelo entienda el contexto completo y genere respuestas más precisas.

In [8]:
def load_documents_from_json(file_path: str) -> List[Document]:
    '''
    Loads documents from a JSON file and transforms them into a list of Document objects.

    Each document is formatted by combining the 'question', 'answer', 'title' and 'text' fields.

    Args:
        file_path (str): Path to the JSON file.

    Returns:
        List[Document]: List of documents ready to be indexed.
    '''
    with open(file_path, 'r', encoding='utf-8') as file:
        data = json.load(file)

    documents: List[Document] = []
    for item in data:
        combined_text = (
            f"Question: {item.get('question', '')}\n"
            f"Answer: {item.get('answer', '')}\n"
            f"Title: {item.get('title', '')}\n"
            f"Text: {item.get('text', '')}"
        )
        documents.append(Document(text=combined_text))
    return documents

Cargamos los documentos del archivo JSON:

In [13]:
documents: List[Document] = load_documents_from_json('hotpotqa_docs_reduced.json')

Inspección básica

In [14]:
len(documents)

1000

In [15]:
documents[0]

Document(id_='c7d58736-23ff-41a4-8d70-5d23926ce6d5', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text="Question: Which magazine was started first Arthur's Magazine or First for Women?\nAnswer: Arthur's Magazine\nTitle: Radio City (Indian radio station)\nText: Radio City is India's first private FM radio station and was started on 3 July 2001.  It broadcasts on 91.1 (earlier 91.0 in most cities) megahertz from Mumbai (where it was started in 2004), Bengaluru (started first in 2001), Lucknow and New Delhi (since 2003).  It plays Hindi, English and regional songs.  It was launched in Hyderabad in March 2006, in Chennai on 7 July 2006 and in Visakhapatnam October 2007.  Radio City recently forayed into New Media in May 2008 with the launch of a music portal - PlanetRadiocity.com that offers music related news

---
### Indexing, Embedding y Almacenamiento

In [22]:
# Configurar los embeddings utilizando un modelo de HuggingFace
embed_model = HuggingFaceEmbedding(model_name='all-MiniLM-L6-v2')

Settings.embed_model = embed_model

In [24]:
# Inicializar el cliente, seteando el path para almacenar los datos
db = chromadb.PersistentClient(path='./chroma_db')

# Crear la colección
chroma_collection = db.get_or_create_collection('hotpotqa_docs')

# Asignar chroma como el vector store por defecto en el contexto
vectore_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vectore_store)

# Crear el index
index = GPTVectorStoreIndex.from_documents(
    documents=documents,
    storage_context=storage_context)

---
### Querying

...`temperature`

In [26]:
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

Settings.llm = OpenAI(
    api_key=OPENAI_API_KEY,
    model='gpt-4o',
    temperature=0)

In [25]:
# Convertir el índice en un query engine
query_engine = index.as_query_engine()

In [28]:
def answer_question(query: str) -> str:
    '''
    Answers a question using the RAG pipeline.

    The function queries the search engine that retrieves the relevant information from the documents and
    uses the GPT-4 model to generate the final answer.

    Args:
        query (str): The question to be answered.

    Returns:
        str: The generated answer.
    '''
    response = query_engine.query(query)
    return str(response)

Haciando consultas:

In [31]:
question = 'Which faith is designated to the University of Providence, private university accredited by the NW association of Schools and Colleges and located in a third largest city in Montana after being passed by Missoula?'
response = answer_question(question)
print('Question: ', question)
print('Response: ', response)

Question:  Which faith is designated to the University of Providence, private university accredited by the NW association of Schools and Colleges and located in a third largest city in Montana after being passed by Missoula?
Response:  Roman Catholic


---
### Evaluación

---
### Referencias

- Documentación oficial de *LlamaIndex*: https://docs.llamaindex.ai/en/stable/
- ¿Qué es RAG?: https://aws.amazon.com/what-is/retrieval-augmented-generation/

--
- `json` — JSON encoder and decoder: https://docs.python.org/3/library/json.html
- `sentence-transformers/all-MiniLM-L6-v2`: https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2